In [ ]:
import torch
import torch.nn as nn 
import torchvision
import torch.optim as optim 
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)), 
    transforms.Grayscale(3), 
    transforms.ToTensor(), 
    transforms.Normalize((0.5, ), (0.5, ))
])

train_loader = DataLoader(torchvision.datasets.MNIST(root='./data', download=True, train=True, transform=transform), batch_size=64, shuffle=True)
test_loader = DataLoader(torchvision.datasets.MNIST(root='./data', download=True, train=False, transform=transform), batch_size=1000)

In [ ]:
class AlexNet(nn.Module): 
    def __init__(self):
        super().__init__()
        self.feats = nn.Sequential(
            nn.Conv2d(3, 64, 11, 4, 2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(64, 192, 5, padding=2), nn.ReLU(), nn.MaxPool2d(3,2),
            nn.Conv2d(192, 384, 3, padding=1), nn.ReLU(), 
            nn.Conv2d(384, 256, 3, padding=1), nn.ReLU(), 
            nn.Conv2d(256,256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(3,2)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(), nn.Linear(256*6*6, 4096), 
            nn.Dropout(), nn.Linear(4096, 4096), nn.ReLU(), 
            nn.Linear(4096,10)
        )
    
    def forward(self, x): 
        x =  self.feats(x)
        x =  x.view(x.size(0), -1)
        x = self.classifier(x)
        return x 

In [ ]:
device = torch.device("cpu")
net = AlexNet().to(device)
crit = nn.CrossEntropyLoss()
opt = optim.Adam(net.parameters(), lr=0.001)

In [ ]:
for e in range(1): 
    net.train()
    r_loss = 0
    
    for i, (im, la) in enumerate(train_loader): 
        
        im, la = im.to(device), la.to(device)
        
        opt.zero_grad()
        loss = crit(net(im), la)
        loss.backward()
        
        opt.step()
        
        r_loss += loss.item()
        
        if i % 100==0: 
            print(f"Epoch: {e+1}, Batch: {i+1}, loss: {r_loss:.3f}")
            r_loss = 0 

In [ ]:
net.eval()
corr = 0
tot = 0 

with torch.no_grad():
    for im, lab in test_loader:
        im, la = im.to(device), la.to(device)
        tot += la.size(0)
        corr += (net(im).argmax(1)==la).sum().item()

print(f"Final Acc: {100*(corr/tot):.2f}")